# Kalshi Markets API - Databricks

Tests the Kalshi API (REST + WebSocket) for college basketball markets (KXNCAAMBGAME).
Uses **kalshi-secrets** scope for credentials. Run **mount_adls** notebook first if writing to Delta.

## Get Unique Title Values

## Setup and Imports

In [11]:
# Databricks: credentials from kalshi-secrets scope (Key Vault)
import json
import pandas as pd
from datetime import datetime

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

In [12]:
# Get Kalshi credentials from Databricks secret scope (Key Vault)
api_key = dbutils.secrets.get(scope="kalshi-secrets", key="kalshi-api-key")
private_key_pem = dbutils.secrets.get(scope="kalshi-secrets", key="kalshi-private-key")

print(f"✓ API Key: {api_key[:8]}...{api_key[-4:]}")
print("✓ Private key: loaded from kalshi-secrets")

✓ API Key: bc5f7913...860b
✓ Private key: path: /Users/mitchleahy/betywety/.secrets


In [13]:
# API Configuration
BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"
ENDPOINT = "/markets"
URL = f"{BASE_URL}{ENDPOINT}"

print(f"Base URL: {BASE_URL}")
print(f"Endpoint: {ENDPOINT}")
print(f"Full URL: {URL}")

Base URL: https://api.elections.kalshi.com/trade-api/v2
Endpoint: /markets
Full URL: https://api.elections.kalshi.com/trade-api/v2/markets


In [14]:
import requests
# this is how we see all the categories that we need to loop through below
url = "https://api.elections.kalshi.com/trade-api/v2/search/tags_by_categories"

response = requests.get(url)

response.text



'{"tags_by_categories":{"Climate and Weather":["Daily temperature","Snow and rain","High temp","Climate change","Natural disasters","Hurricanes"],"Companies":["IPOs","Product launches","KPIs","Elon Musk","CEOs"],"Crypto":["BTC","15 min","Hourly","ETH","SOL","Pre-Market","DOGE","XRP"],"Economics":["Growth","Fed","Inflation","Oil and energy","Employment","Housing"],"Elections":null,"Entertainment":["Music","Movies","Awards","Music charts","Television","Video games","Grammys","Rotten Tomatoes"],"Financials":["S\\u0026P","Nasdaq","Daily","Treasuries","EUR/USD","USD/JPY"],"Mentions":["Politicians","Earnings","Sports"],"Politics":["US Elections","Primaries","Trump","Foreign Elections","International","House","Congress","SCOTUS \\u0026 courts","Local","Recurring"],"Science and Technology":["AI","Space","Energy"],"Social":null,"Sports":["Soccer","Basketball","Baseball","Football","Hockey","Olympics","Esports","Golf","Tennis","MMA","Motorsport","Cricket","Lacrosse","Rugby","Boxing","Chess","Dar

In [22]:
import requests

BASE_URL = "https://api.elections.kalshi.com/trade-api/v2/series"

all_series = []
cursor = None
# test run using basket ball as the tag
while True:
    params = {
        "category": "Sports",
        "tags": "Basketball"
        

    }
    if cursor:
        params["cursor"] = cursor

    resp = requests.get(BASE_URL, params=params)
    resp.raise_for_status()

    data = resp.json()
    all_series.extend(data["series"])

    cursor = data.get("cursor")
    if not cursor:
        break



df_series = pd.DataFrame(all_series)






In [31]:
pd.set_option("display.max_rows", None)
df_series
# Filter rows where the 'title' column contains only 'NBA'
nba_only_rows = df_series[df_series['title'].str.contains('College')]
nba_only_rows



,additional_prohibitions,category,contract_terms_url,contract_url,fee_multiplier,fee_type,frequency,settlement_sources,tags,ticker,title
13,"[Current and former players, coaches, and staf...",Sports,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,custom,"[{'name': 'ESPN', 'url': 'https://www.espn.com...",[Basketball],KXNCAAWBTOTAL,Women's College Basketball Total Points
40,"[Current and former players, coaches, and staf...",Sports,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,annual,"[{'name': 'ESPN', 'url': 'https://www.espn.com...",[Basketball],KXNCAAMBUNDEFEATED,Undefeated in College Basketball Regular Season
64,"[Current and former players, coaches, and staf...",Sports,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic_with_maker_fees,custom,"[{'name': 'ESPN', 'url': 'https://www.espn.com...",[Basketball],KXNCAAMBSPREAD,Men's College Basketball Spread
73,"[Current and former players, coaches, and staf...",Sports,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic_with_maker_fees,custom,"[{'name': 'ESPN', 'url': 'https://www.espn.com...",[Basketball],KXNCAAMBGAME,Men's College Basketball Men's Game
75,"[Current and former players, coaches, and staf...",Sports,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,custom,[{'name': 'the league or association governing...,[Basketball],KXNCAABBSPREAD,College Baseball Spread
100,"[Current and former players, coaches, and staf...",Sports,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,custom,"[{'name': 'ESPN', 'url': 'https://www.espn.com...",[Basketball],KXNCAAWBSPREAD,Women's College Basketball Spread
128,"[Current and former players, coaches, and staf...",Sports,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,custom,"[{'name': 'ESPN', 'url': 'https://www.espn.com...",[Basketball],KXNCAABGAME,College Basketball Game
154,None,Sports,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,annual,"[{'name': 'ESPN', 'url': 'https://www.espn.com...",[Basketball],KXNCAAMBNAISMITH,Men's College Basketball Naismith
159,"[Current and former players, coaches, and staf...",Sports,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic_with_maker_fees,custom,"[{'name': 'AP', 'url': 'https://apnews.com/'},...",[Basketball],KXMARMAD,College Basketball Champion
164,"[Current and former players, coaches, and staf...",Sports,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,custom,"[{'name': 'ESPN', 'url': 'https://www.espn.com...",[Basketball],KXNCAAWBGAME,College Basketball Women's Game


In [49]:
import requests

BASE_URL = "https://api.elections.kalshi.com/trade-api/v2/events"
# looks at the nba ticket 'NBA Team'


series_id = "KXNCAAMBGAME"  # example
all_events = []
cursor = None

while True:
    params = {
        "status": "open",
        "series_ticker": series_id,
        "limit": 200,
    }
    if cursor:
        params["cursor"] = cursor

    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()
    data = r.json()

    all_events.extend(data["events"])
    cursor = data.get("cursor")
    if not cursor:
        break
event_tickers =[]
for event in all_events:
    event_tickers.append(event["event_ticker"])


all_events



[{'available_on_brokers': False,
  'category': 'Sports',
  'collateral_return_type': 'MECNET',
  'event_ticker': 'KXNCAAMBGAME-26FEB17BCFSU',
  'mutually_exclusive': True,
  'product_metadata': {'competition': 'College Basketball (M)',
   'competition_scope': 'Game'},
  'series_ticker': 'KXNCAAMBGAME',
  'strike_period': '',
  'sub_title': 'BC at FSU (Feb 17)',
  'title': 'Boston College at Florida St.'},
 {'available_on_brokers': True,
  'category': 'Sports',
  'collateral_return_type': 'MECNET',
  'event_ticker': 'KXNCAAMBGAME-26FEB16HOUISU',
  'mutually_exclusive': True,
  'product_metadata': {'competition': 'College Basketball (M)',
   'competition_scope': 'Game'},
  'series_ticker': 'KXNCAAMBGAME',
  'strike_period': '',
  'sub_title': 'HOU at ISU (Feb 16)',
  'title': 'Houston at Iowa St.'},
 {'available_on_brokers': True,
  'category': 'Sports',
  'collateral_return_type': 'MECNET',
  'event_ticker': 'KXNCAAMBGAME-26FEB16SOUTXSO',
  'mutually_exclusive': True,
  'product_metadat

In [51]:
import requests
import pandas as pd
import time

BASE = "https://api.elections.kalshi.com/trade-api/v2"

def get_markets_by_event(event_ticker: str, limit: int = 1000):
    markets = []
    cursor = None

    while True:
        
        params = {"limit": limit, "event_ticker": event_ticker}
        if cursor:
            params["cursor"] = cursor

        r = requests.get(f"{BASE}/markets", params=params)
        r.raise_for_status()
        data = r.json()

        markets.extend(data.get("markets", []))
        cursor = data.get("cursor")
        if not cursor:
            break
        

    return markets

all_markets = []
for et in event_tickers:
    all_markets.extend(get_markets_by_event(et))
    time.sleep(0.5)  # Avoid 429 Too Many Requests

df_markets = pd.DataFrame(all_markets)
print("Total markets:", len(df_markets))


Total markets: 114


In [52]:
df_markets



,can_close_early,close_time,created_time,custom_strike,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,fractional_trading_enabled,last_price,last_price_dollars,latest_expiration_time,liquidity,liquidity_dollars,market_type,no_ask,no_ask_dollars,no_bid,no_bid_dollars,no_sub_title,notional_value,notional_value_dollars,open_interest,open_interest_fp,open_time,previous_price,previous_price_dollars,previous_yes_ask,previous_yes_ask_dollars,previous_yes_bid,previous_yes_bid_dollars,price_level_structure,price_ranges,response_price_units,result,rules_primary,rules_secondary,settlement_timer_seconds,status,strike_type,subtitle,tick_size,ticker,title,updated_time,volume,volume_24h,volume_24h_fp,volume_fp,yes_ask,yes_ask_dollars,yes_bid,yes_bid_dollars,yes_sub_title
0,True,2026-03-03T23:00:00Z,2026-02-15T11:04:31.072543Z,{'basketball_team': 'ff86bf01-716c-45e1-9a78-3...,This market will close and expire after a winn...,KXNCAAMBGAME-26FEB17BCFSU,2026-02-18T02:00:00Z,2026-03-03T23:00:00Z,,False,84,0.8400,2026-03-03T23:00:00Z,714297,7142.9700,binary,28,0.2800,15,0.1500,Florida St.,100,1.0000,8,8.00,2026-02-15T16:07:00Z,0,0.0000,0,0.0000,0,0.0000,linear_cent,"[{'end': '1.0000', 'start': '0.0000', 'step': ...",usd_cent,,If Florida St. wins the Boston College at Flor...,The following market refers to the team who wi...,300,active,structured,,1,KXNCAAMBGAME-26FEB17BCFSU-FSU,Boston College at Florida St. Winner?,2026-02-15T18:56:53.59347Z,8,5,5.00,8.00,85,0.8500,72,0.7200,Florida St.
1,True,2026-03-03T23:00:00Z,2026-02-15T11:04:31.072543Z,{'basketball_team': 'b0224585-2413-4e1f-a293-a...,This market will close and expire after a winn...,KXNCAAMBGAME-26FEB17BCFSU,2026-02-18T02:00:00Z,2026-03-03T23:00:00Z,,False,26,0.2600,2026-03-03T23:00:00Z,709045,7090.4500,binary,92,0.9200,74,0.7400,Boston College,100,1.0000,18,18.00,2026-02-15T16:07:00Z,0,0.0000,0,0.0000,0,0.0000,linear_cent,"[{'end': '1.0000', 'start': '0.0000', 'step': ...",usd_cent,,If Boston College wins the Boston College at F...,The following market refers to the team who wi...,300,active,structured,,1,KXNCAAMBGAME-26FEB17BCFSU-BC,Boston College at Florida St. Winner?,2026-02-15T18:56:55.722765Z,18,18,18.00,18.00,26,0.2600,8,0.0800,Boston College
2,True,2026-03-03T02:00:00Z,2026-02-14T14:14:39.6082Z,{'basketball_team': '3e806ca2-b0f4-4840-8165-c...,This market will close and expire after a winn...,KXNCAAMBGAME-26FEB16HOUISU,2026-02-17T05:00:00Z,2026-03-03T02:00:00Z,,False,53,0.5300,2026-03-03T02:00:00Z,560268393,5602683.9300,binary,47,0.4700,44,0.4400,Iowa St.,100,1.0000,2267,2267.00,2026-02-14T19:14:00Z,0,0.0000,0,0.0000,0,0.0000,linear_cent,"[{'end': '1.0000', 'start': '0.0000', 'step': ...",usd_cent,,If Iowa St. wins the Houston at Iowa St. men's...,The following market refers to the team who wi...,300,active,structured,,1,KXNCAAMBGAME-26FEB16HOUISU-ISU,Houston at Iowa St. Winner?,2026-02-15T18:56:46.339077Z,2271,2261,2261.00,2271.00,56,0.5600,53,0.5300,Iowa St.
3,True,2026-03-03T02:00:00Z,2026-02-14T14:14:39.6082Z,{'basketball_team': '91aa26c8-e466-4aed-bc92-f...,This market will close and expire after a winn...,KXNCAAMBGAME-26FEB16HOUISU,2026-02-17T05:00:00Z,2026-03-03T02:00:00Z,,False,48,0.4800,2026-03-03T02:00:00Z,560593853,5605938.5300,binary,56,0.5600,53,0.5300,Houston,100,1.0000,6566,6566.00,2026-02-14T19:14:00Z,0,0.0000,0,0.0000,0,0.0000,linear_cent,"[{'end': '1.0000', 'start': '0.0000', 'step': ...",usd_cent,,If Houston wins the Houston at Iowa St. men's ...,The following market refers to the team who wi...,300,active,structured,,1,KXNCAAMBGAME-26FEB16HOUISU-HOU,Houston at Iowa St. Winner?,2026-02-15T18:56:22.650836Z,6635,6635,6635.00,6635.00,47,0.4700,44,0.4400,Houston
4,True,2026-03-03T01:00:00Z,2026-02-14T14:14:31.055382Z,{'basketball_team': '62d31027-6f8a-412d-80ca-7...,This market will close and expire after a winn...,KXNCAAMBGAME-26FEB16SOUTXSO,2026-02-17T04:00:00Z,2026-03-03T01:00:00Z,,False,54,0.5400,2026-03-03T01:00:00Z,2

In [ ]:
## WebSocket (ticker_v2)

If you get `ModuleNotFoundError: No module named 'websockets'`, run in a new cell: `%pip install websockets`

In [12]:
"""
Kalshi authenticated WebSocket stream (ticker_v2) - Databricks version.
Uses api_key and private_key_pem from kalshi-secrets scope (run credentials cell first).

Prereqs: %pip install websockets (Databricks runtime includes cryptography)
"""

import json
import time
import base64
import asyncio

import websockets
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding

# Credentials from kalshi-secrets (run credentials cell first)
API_KEY_ID = api_key
WS_URL = "wss://api.elections.kalshi.com/trade-api/ws/v2"

# Market tickers from df_markets (run REST cells first)
MARKET_TICKERS = df_markets["ticker"].dropna().astype(str).unique().tolist()


# -------------------------
# Auth helpers
# -------------------------
def load_private_key_from_pem(pem_str: str):
    return serialization.load_pem_private_key(pem_str.encode(), password=None)


def make_ws_headers(api_key_id: str, private_key) -> dict:
    """
    Kalshi WS signature message format:
      timestamp_ms + "GET" + "/trade-api/ws/v2"
    """
    ts_ms = str(int(time.time() * 1000))
    path = "/trade-api/ws/v2"
    message = f"{ts_ms}GET{path}".encode("utf-8")

    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH,
        ),
        hashes.SHA256(),
    )

    return {
        "KALSHI-ACCESS-KEY": api_key_id,
        "KALSHI-ACCESS-TIMESTAMP": ts_ms,
        "KALSHI-ACCESS-SIGNATURE": base64.b64encode(signature).decode("utf-8"),
    }


# -------------------------
# WebSocket client
# -------------------------
async def main():
    if MARKET_TICKERS is None or len(MARKET_TICKERS) == 0:
        raise ValueError("MARKET_TICKERS is empty. Add at least one market ticker to subscribe.")

    private_key = load_private_key_from_pem(private_key_pem)
    headers = make_ws_headers(API_KEY_ID, private_key)

    async with websockets.connect(
        WS_URL,
        additional_headers=headers,
        ping_interval=20,
        ping_timeout=20,
        close_timeout=10,
        max_queue=1024,
    ) as ws:
        sub_msg = {
            "id": 1,
            "cmd": "subscribe",
            "params": {
                "channels": ["ticker_v2"],
                "market_tickers": MARKET_TICKERS,
            },
        }
        await ws.send(json.dumps(sub_msg))
        print(f"Subscribed to ticker_v2 for {len(MARKET_TICKERS)} markets.")

        while True:
            raw = await ws.recv()
            try:
                data = json.loads(raw)
            except json.JSONDecodeError:
                continue

            if data.get("type") == "ticker_v2":
                msg = data.get("msg", {})
                mt = msg.get("market_ticker")
                ts = msg.get("ts")
                yes_bid = msg.get("yes_bid")
                yes_ask = msg.get("yes_ask")
                last_price = msg.get("price")
                print(msg)
            else:
                # Uncomment to see acks/errors:
                # print("NON-TICKER:", data)
                pass


# In Jupyter:
await main()

# In a .py script instead:
# if __name__ == "__main__":
#     asyncio.run(main())




Subscribed to ticker_v2 for 22 markets.
{'market_id': '8003beb7-41f8-446f-a4cc-660628879d4b', 'market_ticker': 'KXNFLANYTD-26FEB08SEANE-NETHENDERSON32', 'yes_bid': 5, 'yes_ask': 9, 'price': 9, 'yes_bid_dollars': '0.0500', 'yes_ask_dollars': '0.0900', 'price_dollars': '0.0900', 'volume_delta': 104, 'volume_delta_fp': '104.00', 'open_interest_delta': 104, 'open_interest_delta_fp': '104.00', 'dollar_volume_delta': 52, 'dollar_open_interest_delta': 52, 'ts': 1770599155}
{'market_id': '6cfe671a-fa04-4ad3-a101-9af52d18c0cb', 'market_ticker': 'KXNFLANYTD-26FEB08SEANE-SEACKUPP10', 'yes_bid': 16, 'yes_ask': 22, 'price': 22, 'yes_bid_dollars': '0.1600', 'yes_ask_dollars': '0.2200', 'price_dollars': '0.2200', 'volume_delta': 430, 'volume_delta_fp': '430.00', 'open_interest_delta': 430, 'open_interest_delta_fp': '430.00', 'dollar_volume_delta': 215, 'dollar_open_interest_delta': 215, 'ts': 1770599155}
{'market_id': '277c2357-4412-4f99-b2e5-4138f4572a64', 'market_ticker': 'KXNFLANYTD-26FEB08SEANE-N

CancelledError: 